In [2]:
import optuna
import pandas as pd
from matplotlib import pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor as DTR
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from math import sqrt

In [3]:
data = pd.read_csv('../data/processed_trip_duration.csv')
X = data.drop(['trip_duration'], axis=1)
y = data['trip_duration']
print(X.shape, y.shape)

(655757, 33) (655757,)


In [5]:
def objective(trial):
    criterion = trial.suggest_categorical('criterion', ['absolute_error', 'friedman_mse', 'squared_error', 'poisson'])
    splitter = trial.suggest_categorical('splitter', ['best', 'random'])
    max_depth = trial.suggest_int('max_depth', 1, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])


    model = DTR(criterion=criterion,
                splitter=splitter,
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features)
    
    score = cross_val_score(model, X, y, cv=4, scoring='neg_mean_squared_error', n_jobs=1).mean()

    return score

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, n_jobs=-1)

[I 2025-04-21 14:11:01,824] A new study created in memory with name: no-name-af3ea30c-0d98-4352-973a-003e57d511e1
[I 2025-04-21 14:11:09,222] Trial 3 finished with value: -201070.68846141433 and parameters: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 2, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 3 with value: -201070.68846141433.
[I 2025-04-21 14:11:12,856] Trial 0 finished with value: -181146.41157545775 and parameters: {'criterion': 'poisson', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: -181146.41157545775.
[I 2025-04-21 14:11:13,644] Trial 5 finished with value: -175014.4890455858 and parameters: {'criterion': 'poisson', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 5 with value: -175014.4890455858.
[I 2025-04-21 14:11:14,697] Trial 2 finished with value:

In [ ]:
print('Best params', study.best_params)
print('Best MSE', study.best_value)

Trial 18 finished with value: -82502.33010761077 and parameters: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 15, 'min_samples_split': 6, 'min_samples_leaf': 7, 'max_features': None}.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
)

In [5]:
dtr = DTR(
    criterion='squared_error',
    splitter= 'best',
    max_depth=15,
    min_samples_split=6,
    min_samples_leaf=7)

In [6]:
dtr.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=15, min_samples_leaf=7, min_samples_split=6)

In [7]:
y_pred = dtr.predict(X_test)

In [8]:
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {sqrt(mean_squared_error(y_test, y_pred))}')
print(f'MAPE: {sqrt(mean_absolute_percentage_error(y_test, y_pred))}')
print(f'R^2: {round(r2_score(y_test, y_pred),2)}')

MAE: 204.45272068553248
MSE: 82472.2827619509
RMSE: 287.1798787553733
MAPE: 0.5840161143092989
R^2: 0.64


In [9]:
text_representation = tree.export_text(dtr)
print(text_representation)

|--- feature_6 <= 3029.05
|   |--- feature_6 <= 1565.36
|   |   |--- feature_6 <= 1070.04
|   |   |   |--- feature_6 <= 151.75
|   |   |   |   |--- feature_6 <= 88.71
|   |   |   |   |   |--- feature_6 <= 6.17
|   |   |   |   |   |   |--- feature_3 <= -73.93
|   |   |   |   |   |   |   |--- feature_6 <= 1.27
|   |   |   |   |   |   |   |   |--- feature_26 <= 0.50
|   |   |   |   |   |   |   |   |   |--- feature_20 <= 0.50
|   |   |   |   |   |   |   |   |   |   |--- feature_1 <= -74.01
|   |   |   |   |   |   |   |   |   |   |   |--- value: [1037.14]
|   |   |   |   |   |   |   |   |   |   |--- feature_1 >  -74.01
|   |   |   |   |   |   |   |   |   |   |   |--- truncated branch of depth 5
|   |   |   |   |   |   |   |   |   |--- feature_20 >  0.50
|   |   |   |   |   |   |   |   |   |   |--- feature_3 <= -73.96
|   |   |   |   |   |   |   |   |   |   |   |--- truncated branch of depth 4
|   |   |   |   |   |   |   |   |   |   |--- feature_3 >  -73.96
|   |   |   |   |   |   |   |   | 